# Phase 0.4 & 0.5 — AI Search endpoint + Delta Sync index

**Where to run:** Databricks serverless notebook (recommended), or locally in Cursor with CLI profile `dbx-lzago-ai`.

**Prerequisite:** `workspace.document_retrieval.document_chunks` exists (Phase 0.3).

| Object | Name |
|---|---|
| Endpoint | `document-chunks-search-endpoint` |
| Source table | `workspace.document_retrieval.document_chunks` |
| Index | `workspace.document_retrieval.document_chunks_index` |

In [9]:
# On Databricks, %pip restarts the Python kernel automatically after install.
%pip install -q databricks-ai-search databricks-connect openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import time
from pathlib import Path

from databricks.ai_search.client import AISearchClient
from databricks.ai_search.exceptions import InvalidInputException
from databricks.sdk import WorkspaceClient
from databricks.sdk.core import Config
from dotenv import load_dotenv
from openai import OpenAI

CATALOG = "workspace"
SCHEMA = "document_retrieval"
TABLE = "document_chunks"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"
ENDPOINT_NAME = "document-chunks-search-endpoint"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}_index"
EMBEDDING_DIMENSION = 1536  # text-embedding-3-small
EMBEDDING_MODEL = "text-embedding-3-small"
PROFILE = os.getenv("DATABRICKS_CONFIG_PROFILE", "dbx-lzago-ai")


def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "main.py").exists():
            return path
    return Path.cwd()


load_dotenv(find_repo_root() / ".env")


def get_openai_api_key() -> str:
    key = os.getenv("OPENAI_API_KEY")
    if key:
        return key
    try:
        return dbutils.secrets.get("week1v2", "openai_api_key")  # type: ignore[name-defined]
    except NameError:
        pass
    raise RuntimeError(
        "OPENAI_API_KEY not set. For local runs: cp .env.example .env and add your key. "
        "On Databricks: use secret scope week1v2 / key openai_api_key."
    )


def get_openai_client() -> OpenAI:
    return OpenAI(api_key=get_openai_api_key())


def get_ai_search_client() -> AISearchClient:
    """Use notebook auto-auth on Databricks; CLI OAuth profile when running locally."""
    try:
        return AISearchClient()
    except InvalidInputException:
        cfg = Config(profile=PROFILE)
        token = cfg.authenticate()["Authorization"].removeprefix("Bearer ").strip()
        return AISearchClient(workspace_url=cfg.host, personal_access_token=token)


def endpoint_state(response: dict) -> str:
    """Read endpoint state from SDK or CLI response shapes."""
    payload = response.get("endpoint", response)
    status = payload.get("endpoint_status", "UNKNOWN")
    if isinstance(status, dict):
        return status.get("state", "UNKNOWN")
    return str(status)


def get_spark():
    """Use Databricks notebook spark, or Databricks Connect serverless locally."""
    try:
        return spark  # type: ignore[name-defined]  # injected on Databricks
    except NameError:
        from databricks.connect import DatabricksSession

        return DatabricksSession.builder.profile(PROFILE).serverless().getOrCreate()


def get_workspace_client() -> WorkspaceClient:
    """Preferred client for index status, sync, and query (works with CLI OAuth)."""
    return WorkspaceClient(profile=PROFILE)


def index_status(desc: dict) -> tuple[bool, str]:
    status = desc.get("status", {})
    if isinstance(status, dict):
        return bool(status.get("ready")), status.get("detailed_state", "UNKNOWN")
    return False, "UNKNOWN"


def wait_for_index_ready(attempts: int = 60, sleep_seconds: int = 20):
    """Poll until status.ready is true (can take several minutes after create)."""
    w = get_workspace_client()
    for attempt in range(attempts):
        idx = w.vector_search_indexes.get_index(index_name=INDEX_NAME)
        ready = bool(idx.status and idx.status.ready)
        row_count = idx.status.indexed_row_count if idx.status else None
        message = idx.status.message if idx.status else ""
        print(f"  [{attempt + 1}/{attempts}] ready={ready} rows={row_count}")
        if message:
            print(f"    {message}")
        if ready:
            return idx
        time.sleep(sleep_seconds)
    raise RuntimeError(f"Index not ready after waiting: {INDEX_NAME}")


client = get_ai_search_client()
print("Source table:", SOURCE_TABLE)
print("Endpoint:    ", ENDPOINT_NAME)
print("Index:       ", INDEX_NAME)
print("OpenAI key:  ", end="")
try:
    get_openai_api_key()
    print("configured")
except RuntimeError as exc:
    print(f"MISSING — {exc}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Source table: workspace.document_retrieval.document_chunks
Endpoint:     document-chunks-search-endpoint
Index:        workspace.document_retrieval.document_chunks_index
OpenAI key:  configured


## Phase 0.4 — Create endpoint

In [9]:
response = client.list_endpoints()
raw_endpoints = response.get("endpoints", response if isinstance(response, list) else [])
existing = [ep.get("name", ep) if isinstance(ep, dict) else ep for ep in raw_endpoints]

if ENDPOINT_NAME in existing:
    print(f"Endpoint already exists: {ENDPOINT_NAME}")
else:
    print(f"Creating endpoint: {ENDPOINT_NAME}")
    client.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    print("Create request sent.")

for attempt in range(30):
    status = client.get_endpoint(name=ENDPOINT_NAME)
    state = endpoint_state(status)
    print(f"  [{attempt + 1}/30] status: {state}")
    if state == "ONLINE":
        break
    time.sleep(20)
else:
    raise RuntimeError(f"Endpoint not ONLINE after waiting: {ENDPOINT_NAME}")

print("Phase 0.4 complete.")

Endpoint already exists: document-chunks-search-endpoint
  [1/30] status: ONLINE
Phase 0.4 complete.


## Phase 0.5 — Create Delta Sync index

Uses **existing embeddings** in column `text_vector` (1536 dims). Sync mode: **TRIGGERED**.

In [10]:
try:
    index = client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    print(f"Index already exists: {INDEX_NAME}")
except Exception:
    print(f"Creating Delta Sync index: {INDEX_NAME}")
    index = client.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        source_table_name=SOURCE_TABLE,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_dimension=EMBEDDING_DIMENSION,
        embedding_vector_column="text_vector",
        columns_to_sync=[
            "document_id",
            "chunk_index",
            "chunk_text",
            "source",
        ],
    )
    print("Create request sent.")

index = client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
print("Waiting for index to become ready (this can take a few minutes)...")
wait_for_index_ready()
print("Phase 0.5 complete.")

Creating Delta Sync index: workspace.document_retrieval.document_chunks_index
Create request sent.
{'name': 'workspace.document_retrieval.document_chunks_index', 'endpoint_name': 'document-chunks-search-endpoint', 'primary_key': 'id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'workspace.document_retrieval.document_chunks', 'embedding_vector_columns': [{'name': 'text_vector', 'embedding_dimension': 1536}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': '7c840c2c-ee35-42cc-82db-d00e36516157'}, 'status': {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta sync index creation is pending endpoint provisioning.', 'ready': False, 'index_url': 'd6db9823-2c83-47fa-9ba9-cfff663db7b8.vector-search.cloud.databricks.com/api/2.0/299177927171866/vector-search/indexes/workspace.document_retrieval.document_chunks_index'}, 'creator': 'lukaszago@hotmail.com', 'endpoint_type': 'STANDARD', 'id': 'dac0eb7d-e1c6-4cf5-8a15-bbbf271e6de5', 'index_subtype': 'HYBRID', 'endpoint

## Phase 0.7 — Insert test row, sync, and query

Requires `OPENAI_API_KEY` in repo-root `.env` (local) or secret `week1v2` / `openai_api_key` (Databricks).

In [3]:
spark = get_spark()
oai = get_openai_client()

text = "Remote work: up to 3 days per week with manager approval."
vector = oai.embeddings.create(model=EMBEDDING_MODEL, input=[text]).data[0].embedding
assert len(vector) == EMBEDDING_DIMENSION, f"Expected {EMBEDDING_DIMENSION}, got {len(vector)}"

spark.createDataFrame([{
    "id": "handbook::0",
    "document_id": "handbook",
    "chunk_index": 0,
    "chunk_text": text,
    "source": "handbook",
    "text_vector": vector,
}]).write.mode("append").saveAsTable(SOURCE_TABLE)

print("Inserted 1 row into", SOURCE_TABLE)

Inserted 1 row into workspace.document_retrieval.document_chunks


## Phase 0.7 preview — initial sync (run after inserting test data)

Skip this cell until you have at least one row in the source table.

In [12]:
spark = get_spark()
row_count = spark.table(SOURCE_TABLE).count()
print(f"Source table rows: {row_count}")

if row_count == 0:
    print("Table is empty — insert a test row first, then re-run this cell.")
else:
    w = get_workspace_client()
    print("Checking index is ready before sync...")
    wait_for_index_ready()
    w.vector_search_indexes.sync_index(index_name=INDEX_NAME)
    print("Sync triggered. Wait 1–2 minutes before querying.")

Source table rows: 1
Checking index is ready before sync...
  [1/60] ready=True rows=1
    Index creation succeeded. Check latest status: https://dbc-d3858b75-976f.cloud.databricks.com/explore/data/workspace/document_retrieval/document_chunks_index
Sync triggered. Wait 1–2 minutes before querying.


In [13]:
oai = get_openai_client()
w = get_workspace_client()

question = "What is the remote work policy?"
q_vector = oai.embeddings.create(model=EMBEDDING_MODEL, input=[question]).data[0].embedding

results = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    columns=["id", "document_id", "chunk_text"],
    query_vector=q_vector,
    num_results=5,
)
print(results.result.data_array)
print("Phase 0.7 complete if handbook chunk is in top results.")

[['handbook::0', 'handbook', 'Remote work: up to 3 days per week with manager approval.', 0.53327096]]
Phase 0.7 complete if handbook chunk is in top results.
